# MediPredict AI: Exploratory Data Analysis & Clinical ML Lifecycle
**Course:** BTech Artificial Intelligence and Data Science - Machine Learning Capstone
**Dataset:** Pima Indians Diabetes Database (National Institute of Diabetes and Digestive and Kidney Diseases)

### Abstract
This notebook documents the exploratory data analysis, physiological zero inspection, leakage-free preprocessing pipeline, and initial model evaluation methodology for MediPredict AI.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_raw_data
from src.data.validate_data import generate_data_quality_report
from src.features.build_features import FEATURE_NAMES, TARGET_NAME, SUSPICIOUS_ZERO_COLUMNS

print("Environment initialized successfully.")

## 1. Load Raw Dataset and Verify Provenance (SHA-256)

In [ ]:
raw_df, sha256_hash = load_raw_data(data_path=PROJECT_ROOT / 'data' / 'raw' / 'diabetes.csv')
print(f"Dataset Shape: {raw_df.shape}")
print(f"SHA-256 Checksum: {sha256_hash}")
raw_df.head()

## 2. Data Quality Audit & Physiological Zero Inspection
In living humans, measurements such as plasma glucose, blood pressure, insulin, and BMI cannot physically be zero.
Rather than deleting almost 50% of patient records, we treat these zeroes as missing values.

In [ ]:
quality_report = generate_data_quality_report(raw_df)
zero_summary = pd.DataFrame(quality_report["suspicious_zero_summary"]).T
zero_summary[["zero_count", "percentage", "clinical_reason"]]

## 3. Class Balance and Target Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=raw_df, x=TARGET_NAME, palette=["#0f766e", "#dc2626"])
plt.title("Class Distribution: Outcome (0 = Non-diabetic, 1 = Diabetic)")
plt.xlabel("Diabetes Outcome")
plt.ylabel("Patient Count")
plt.show()

print("Class Proportions:")
print(raw_df[TARGET_NAME].value_counts(normalize=True).round(3))

## 4. Preprocessing & Leakage-Free Pipeline Verification

In [ ]:
from src.data.preprocess_data import split_data, create_preprocessor_pipeline

X_train, X_test, y_train, y_test = split_data(raw_df, test_size=0.20, random_state=42)
preprocessor = create_preprocessor_pipeline(with_scaling=True)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"X_train shape: {X_train_proc.shape}")
print(f"X_test shape: {X_test_proc.shape}")
print("Learned Median Imputation Values:", preprocessor.named_steps['imputer'].statistics_)

## 5. Model Evaluation Leaderboard

In [ ]:
csv_path = PROJECT_ROOT / 'reports' / 'evaluation_summary.csv'
if csv_path.exists():
    results_df = pd.read_csv(csv_path)
    display(results_df)
else:
    print("Run 'python -m src.training.train' to generate benchmark results.")